# Dokumente gezielt nach Fabric aktualisieren


In [1]:
PROJECT_ROOT="$(git rev-parse --show-toplevel)" && cd "$PROJECT_ROOT" &&
SOURCE="data/pdf_sources/P-200_offer.txt" &&
PDF="data/pdfs/P-200_offer.pdf" &&
DOC_ID="P-200_offer" &&
CSV_TABLE="emails" &&
IMPORT_CSV=0 && IMPORT_SEMANTIC=0 && IMPORT_ONTOLOGY=0 &&
printf "Projekt: %s\nPDF: %s\n" "$PROJECT_ROOT" "$PDF"

Projekt: /home/ga1/cptdazagentprojectmanagment


PDF: data/pdfs/P-200_offer.pdf


## Ablauf und Grenzen

Dieses Notebook rendert eine einfache Demo-PDF aus bearbeitbarem Text, analysiert nur diese PDF mit CU Document Routing und aktualisiert ihre Zeile anhand der DocId. Die anderen Dokumentzeilen werden aus einem frisch aktualisierten semantischen Modell gelesen und erhalten. Anschliessend folgen OneLake-Upload, Tabellenladen, Modell-Refresh und ein Vergleich aller Tabellenwerte.

Die Standardausfuehrung schreibt nur die Dokumenttabelle und aktualisiert das Modell. Agent, Ontologie-Definition und andere Tabellen bleiben unveraendert. Keine parallelen Imports oder anderen Schreiber auf der Dokumenttabelle ausfuehren: der Einzelimport ist ein Read-Merge-Overwrite, keine transaktionale Zeilenoperation. Die Demo-PDF-Erzeugung ist kein Editor fuer beliebig formatierte Original-PDFs.

Voraussetzungen: az login, vorhandenes Deployment, laufende Fabric Capacity und die Abhaengigkeiten aus scripts/requirements.txt. Kernel: Bash aus .venv. Die drei optionalen Importaktionen bleiben bei Run All ausgeschaltet, bis ihre Schalter in der ersten Zelle auf 1 gesetzt werden.

In [2]:
AZURE_RESOURCE_GROUP="${AZURE_RESOURCE_GROUP:-rg-projplan-demo}" &&
deployment_output() { az deployment group show -g "$AZURE_RESOURCE_GROUP" -n main --query "properties.outputs.$1.value" -o tsv; } &&
FOUNDRY_ACCOUNT_NAME="$(deployment_output foundryAccountName)" &&
FOUNDRY_MODEL_NAME="$(deployment_output modelDeployment)" &&
FOUNDRY_EMBEDDING_NAME="$(deployment_output embeddingDeployment)" &&
export FOUNDRY_ACCOUNT_NAME FOUNDRY_MODEL_NAME FOUNDRY_EMBEDDING_NAME &&
az account show --query "{subscription:id}" -o json

{


  "subscription": "98525264-1eb4-493f-983d-16a330caa7f6"


}


## Aktuellen Stand sichern

Die folgende Abfrage liest das semantische Modell, ohne Daten zu veraendern. Der Snapshot dient nach dem Import zum Vergleich der nicht bearbeiteten Dokumente.

In [3]:
( set -euo pipefail
mkdir -p extraction/outputs
.venv/bin/python scripts/import_fabric.py --target status --table documents > extraction/outputs/notebook_documents_before.json
jq ".rows | map({DocId, DocType, Title})" extraction/outputs/notebook_documents_before.json
)

[


  {


    "DocId": "P-100_contract",


    "DocType": "Contract",


    "Title": "Construction Contract - Office Building Hamburg (P-100)"


  },


... (17 outputs left out)

    "DocType": "Contract",


    "Title": "Offer - Residential Complex Munich (P-200)"


  }


]


## Dokumenttext bearbeiten

Aendere den Text in der folgenden Zelle und fuehre ab hier erneut aus. Fuer den ersten Test kann beispielsweise die Revision von notebook-demo-v2 auf notebook-demo-v3 geaendert werden. Dateiname und Ueberschrift bleiben absichtlich Offer, waehrend der Inhalt einen Vertrag beschreibt.

Die Zelle ueberschreibt die ausgewaehlte Textquelle. Aenderungen an Budget oder Terminen werden ebenfalls importiert und koennen bewusst von den Demo-Sollwerten abweichen. Die Sollwerte sind nur ein Vergleich, keine Ersatzdaten.

In [4]:
( set -euo pipefail
cat > "$SOURCE" <<'DOCUMENT'
Offer - Residential Complex Munich (P-200)
Parties: Demo Construction AG and Isar Housing Corp
The parties hereby enter into this binding construction contract.
This signed agreement is effective immediately and is not a quotation.
The contractor shall perform the works; the client shall pay the agreed budget.
Budget: EUR 5,000,000. An overrun of more than 10 % requires written approval by the client.
Term: 2027-03-01 to 2028-06-30.
Both parties accept these contractual obligations.
Signed for Demo Construction AG: Demo Contractor
Signed for Isar Housing Corp: Demo Client
Revision: notebook-demo-v2.
DOCUMENT
.venv/bin/python scripts/create_demo_pdf.py "$SOURCE" "$PDF"
cat "$SOURCE"
)

Created data/pdfs/P-200_offer.pdf (1 pages) from data/pdf_sources/P-200_offer.txt


Offer - Residential Complex Munich (P-200)


Parties: Demo Construction AG and Isar Housing Corp


The parties hereby enter into this binding construction contract.


This signed agreement is effective immediately and is not a quotation.


The contractor shall perform the works; the client shall pay the agreed budget.


... (2 outputs left out)

Both parties accept these contractual obligations.


Signed for Demo Construction AG: Demo Contractor


Signed for Isar Housing Corp: Demo Client


Revision: notebook-demo-v2.


## Nur diese PDF importieren und Refresh abwarten

Die gemeinsame CLI analysiert die Datei, speichert das vollstaendige CU-Ergebnis lokal, erhaelt die anderen Dokumentzeilen, laedt die zusammengefuehrte Dokument-CSV und wartet auf Refresh und Inhaltspruefung. Sie erstellt keine neue Agent-Version. Bei einem Fehler stoppt diese Zelle mit einer sichtbaren Fehlermeldung.

In [5]:
( set -euo pipefail
.venv/bin/python scripts/import_fabric.py --target pdfs --file "$PDF"
jq . "extraction/outputs/$DOC_ID.json"
jq . extraction/outputs/import_report.json
)

  Classified P-200_offer.pdf: Contract


  Extraction: 12 of 12 fields match the expected values


  Content Understanding outputs saved to /home/ga1/cptdazagentprojectmanagment/extraction/outputs


Semantic model refresh: e15c4afd-c645-49fa-8671-f9aaa5708854


Semantic model refresh completed.


Merged P-200_offer; preserved 4 other documents.


... (1,366 outputs left out)

  "verified_rows": {


    "documents": 5


  }


}


## Sichtbaren Inhalt und unveraenderte Dokumente pruefen

Die Import-CLI hat bereits alle Werte der gesamten Dokumenttabelle gegen die hochgeladene CSV verglichen. Zusaetzlich pruefen wir hier, dass die uebrigen Dokumente seit dem ersten Snapshot unveraendert geblieben sind. Fuer diese Demo muss das falsch betitelte Dokument als Contract sichtbar sein.

In [6]:
( set -euo pipefail
.venv/bin/python scripts/import_fabric.py --target status --table documents > extraction/outputs/notebook_documents_after.json
jq --arg id "$DOC_ID" ".rows | map(select(.DocId == \$id))" extraction/outputs/notebook_documents_after.json
jq -e --arg id "$DOC_ID" --arg revision "$(tail -n 1 "$SOURCE")" --slurpfile before extraction/outputs/notebook_documents_before.json '
  ((.rows | map(select(.DocId != $id)) | sort_by(.DocId)) ==
   ($before[0].rows | map(select(.DocId != $id)) | sort_by(.DocId))) and
  ([.rows[] | select(.DocId == $id)] | length == 1) and
  all(.rows[] | select(.DocId == $id); .DocType == "Contract" and (.Content | contains($revision)))
' extraction/outputs/notebook_documents_after.json
)

[


  {


    "DocId": "P-200_offer",


    "ProjectId": "P-200",


    "DocType": "Contract",


    "Title": "Offer - Residential Complex Munich (P-200)",


... (12 outputs left out)

    "OverrunApprovalThresholdPercent": 10.0


  }


]


true


## Optionale, getrennte Importaktionen

Die folgenden Zellen sind unabhaengig vom Einzel-PDF-Update. Aktiviere nur die benoetigte Aktion ueber ihren Schalter in der Konfigurationszelle. CSV ersetzt die ausgewaehlte Tabelle vollstaendig und analysiert keine PDFs. Semantic veroeffentlicht nur die Modell-Definition, aktualisiert SQL-Metadaten und wartet auf Modell-Refresh. Ontology veroeffentlicht nur die Ontologie-Definition. Ohne geaenderte Definition ist eine erneute Veroeffentlichung fuer reine Datenupdates nicht erforderlich.

In [7]:
if [[ "$IMPORT_CSV" == 1 ]]; then
  ( set -euo pipefail; .venv/bin/python scripts/import_fabric.py --target csv --table "$CSV_TABLE" )
else
  printf "CSV-Import deaktiviert. Fuer gezielten Import IMPORT_CSV=1 setzen.\n"
fi

CSV-Import deaktiviert. Fuer gezielten Import IMPORT_CSV=1 setzen.


In [8]:
if [[ "$IMPORT_SEMANTIC" == 1 ]]; then
  ( set -euo pipefail; .venv/bin/python scripts/import_fabric.py --target semantic )
else
  printf "Modell-Definitionsimport deaktiviert. Fuer Import IMPORT_SEMANTIC=1 setzen.\n"
fi

Modell-Definitionsimport deaktiviert. Fuer Import IMPORT_SEMANTIC=1 setzen.


In [9]:
if [[ "$IMPORT_ONTOLOGY" == 1 ]]; then
  ( set -euo pipefail; .venv/bin/python scripts/import_fabric.py --target ontology )
else
  printf "Ontologie-Definitionsimport deaktiviert. Fuer Import IMPORT_ONTOLOGY=1 setzen.\n"
fi

Ontologie-Definitionsimport deaktiviert. Fuer Import IMPORT_ONTOLOGY=1 setzen.
